In [1]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      40
  On-line CPU(s) list:       0
  Off-line CPU(s) list:      1-39
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) Gold 5115 CPU @ 2.40GHz
    CPU family:              6
    Model:                   85
    Thread(s) per core:      2
    Core(s) per socket:      10
    Socket(s):               2
    Stepping:                4
    BogoMIPS:                4800.00
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush dts acpi mmx fxsr sse 
                             sse2 ss ht tm pbe syscall nx pdpe1gb rdtscp lm cons
                             tant_tsc art arch_perfmon pebs bts rep_good nopl xt
                             opology nonstop_tsc cpuid a

In [3]:
import os
os.environ["PATH"] += os.pathsep + "/usr/local/bin"
!nasm -v

NASM version 2.16.03 compiled on Nov 28 2024


In [11]:
%%writefile asmfunc3.asm
global asmfunc3

section .text

asmfunc3:
    mov     r9, rdi
    xor     rax, rax

    mov     r10, r9
    sub     r10, 8

vector_loop:
    cmp     rax, r10
    jg      remainder

    vmovaps ymm0, [rsi + rax*4]
    vmovaps ymm1, [rdx + rax*4]

    vmaxps  ymm2, ymm0, ymm1
    vmovaps [rcx + rax*4], ymm2

    ; idx = 1 when A < B
    vcmpps  ymm3, ymm0, ymm1, 1
    vpsrld  ymm3, ymm3, 31
    vmovdqu [r8 + rax*4], ymm3

    add     rax, 8
    jmp     vector_loop

remainder:
    cmp     rax, r9
    jge     done

scalar_loop:
    movss   xmm0, [rsi + rax*4]
    movss   xmm1, [rdx + rax*4]

    ucomiss xmm0, xmm1
    jae     takeA

takeB:
    movss   [rcx + rax*4], xmm1
    mov     dword [r8 + rax*4], 1
    jmp     next

takeA:
    movss   [rcx + rax*4], xmm0
    mov     dword [r8 + rax*4], 0

next:
    inc     rax
    cmp     rax, r9
    jl      scalar_loop

done:
    vzeroupper
    ret

Overwriting asmfunc3.asm


In [12]:
%%writefile asmfunc2.asm
global asmfunc2

section .text

asmfunc2:
    mov     r9, rdi
    xor     rax, rax

    mov     r10, r9
    sub     r10, 4

vector_loop:
    cmp     rax, r10
    jg      remainder

    movaps  xmm0, [rsi + rax*4]
    movaps  xmm1, [rdx + rax*4]

    movaps  xmm0, [rsi + rax*4]    
    movaps  xmm1, [rdx + rax*4]   

    movaps  xmm2, xmm0
    maxps   xmm2, xmm1
    movaps  [rcx + rax*4], xmm2

    movaps  xmm3, xmm0             
    cmpps   xmm3, xmm1, 1         
    psrld   xmm3, 31
    movdqu  [r8 + rax*4], xmm3

    add     rax, 4
    jmp     vector_loop

remainder:
    cmp     rax, r9
    jge     done

scalar_loop:
    movss   xmm0, [rsi + rax*4]
    movss   xmm1, [rdx + rax*4]

    ucomiss xmm0, xmm1
    jae     takeA

takeB:
    movss   [rcx + rax*4], xmm1
    mov     dword [r8 + rax*4], 1
    jmp     next

takeA:
    movss   [rcx + rax*4], xmm0
    mov     dword [r8 + rax*4], 0

next:
    inc     rax
    cmp     rax, r9
    jl      scalar_loop

done:
    ret

Overwriting asmfunc2.asm


In [13]:
%%writefile asmfunc1.asm
global asmfunc1

section .text

asmfunc1:
    xor r9d, r9d

.loop:
    cmp r9d, edi
    jge .done

    movss xmm0, [rsi + r9*4]
    movss xmm1, [rdx + r9*4]

    ucomiss xmm0, xmm1
    jae .takeA

.takeB:
    movss [rcx + r9*4], xmm1
    mov dword [r8 + r9*4], 1
    jmp .next

.takeA:
    movss [rcx + r9*4], xmm0
    mov dword [r8 + r9*4], 0

.next:
    inc r9d
    jmp .loop

.done:
    ret

Overwriting asmfunc1.asm


In [4]:
%%writefile main.c

#include <stdio.h>
#include <stdlib.h>
#include <time.h>

extern void asmfunc1(int n, float A[], float B[], float C[], int idx[]);
extern void asmfunc2(int n, float A[], float B[], float C[], int idx[]);
extern void asmfunc3(int n, float A[], float B[], float C[], int idx[]);

void c_kernel(int n, float A[], float B[], float C[], int idx[])
{
    for (int i = 0; i < n; i++)
    {
        if (A[i] >= B[i])
        {
            C[i] = A[i];
            idx[i] = 0;
        }
        else
        {
            C[i] = B[i];
            idx[i] = 1;
        }
    }
}

int check_correctness(int n,float C_ref[],int idx_ref[],float C_test[], int idx_test[])
{
    for (int i = 0; i < n; i++)
    {
        if (C_ref[i] != C_test[i])
        {
            printf("C mismatch at %d\n", i);
            printf("Expected %.2f Got %.2f\n", C_ref[i], C_test[i]);
            return 0;
        }
        if (idx_ref[i] != idx_test[i])
        {
            printf("idx mismatch at %d\n", i);
            printf("Expected %d Got %d\n",idx_ref[i], idx_test[i]);
            return 0;
        }
    }
    return 1;
}
void print_results(const char *name, int n, float C[], int idx[])
{
    printf("\n%s\n", name);
    printf("First 5 elements:\n");
    for (int i = 0; i < 5; i++)
    {
        printf("[%d] C = %.2f\tidx = %d\n", i, C[i], idx[i]);
    }

    printf("\nLast 5 elements:\n");
    for (int i = n - 5; i < n; i++)
    {
        printf("[%d] C = %.2f\tidx = %d\n", i, C[i], idx[i]);
    }

    printf("\n");
}

double get_time()
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec / 1000000000.0;
}

double benchmark( void (*kernel)(int, float[], float[], float[], int[]), int n,float A[],float B[],float C[], int idx[])
{
    double total = 0.0;
    kernel(n, A, B, C, idx);
    double start = get_time();
    kernel(n, A, B, C, idx);
    double end = get_time();
    total = end - start;
    return total;
}

int main()
{
    size_t sizes[] = {
        1ULL << 20,
        1ULL << 26,
        1ULL << 30
    };

    int num_sizes = 3;

    for (int s = 0; s < num_sizes; s++)
    {
        int n = sizes[s];

        printf("\n====================================\n");
        printf("N = %d elements\n", n);
        printf("====================================\n");

        float *A;
        float *B;

        float *C;
        float *C_ref;

        int *idx;
        int *idx_ref;

        if (posix_memalign((void**)&A, 32, (size_t)n * sizeof(float)) != 0 ||
            posix_memalign((void**)&B, 32, (size_t)n * sizeof(float)) != 0 ||
            posix_memalign((void**)&C, 32, (size_t)n * sizeof(float)) != 0 ||
            posix_memalign((void**)&C_ref, 32, (size_t)n * sizeof(float)) != 0 ||
            posix_memalign((void**)&idx, 32, (size_t)n * sizeof(int)) != 0 ||
            posix_memalign((void**)&idx_ref, 32, (size_t)n * sizeof(int)) != 0)
        {
            printf("Memory allocation failed!\n");
            return 1;
        }

        for (int i = 0; i < n; i++)
        {
            A[i] = (float)(i % 1000);
            B[i] = (float)(999 - (i % 1000));
        }

        
        c_kernel(n, A, B, C_ref, idx_ref);
        print_results("C Kernel", n, C_ref, idx_ref);
        asmfunc1(n, A, B, C, idx);
        printf("Scalar : %s\n",check_correctness(n, C_ref, idx_ref, C, idx) ? "PASS" : "FAIL");
        print_results("Scalar", n, C, idx);
        asmfunc2(n, A, B, C, idx);
        printf("XMM    : %s\n", check_correctness(n, C_ref, idx_ref, C, idx) ? "PASS" : "FAIL");
        print_results("XMM", n, C, idx);
        asmfunc3(n, A, B, C, idx);
        printf("YMM    : %s\n", check_correctness(n, C_ref, idx_ref, C, idx) ? "PASS" : "FAIL");
        print_results("YMM", n, C, idx);
        
        double c_time = benchmark(c_kernel, n, A, B, C, idx);
        double scalar_time = benchmark(asmfunc1, n, A, B, C, idx);
        double xmm_time = benchmark(asmfunc2, n, A, B, C, idx);
        double ymm_time = benchmark(asmfunc3, n, A, B, C, idx);

        printf("\n Execution Time:\n");
        printf("C      : %.8f MS\n", c_time * 1000.0);
        printf("Scalar : %.8f MS\n", scalar_time* 1000.0);
        printf("XMM    : %.8f MS\n", xmm_time* 1000.0);
        printf("YMM    : %.8f MS\n", ymm_time* 1000.0);
        printf("\nSpeedup compared to Scalar:\n");
        printf("XMM : %.2fx\n", scalar_time / xmm_time);
        printf("YMM : %.2fx\n", scalar_time / ymm_time);

        free(A);
        free(B);
        free(C);
        free(C_ref);
        free(idx);
        free(idx_ref);
    }

    return 0;
}

Overwriting main.c


In [5]:
!nasm -f elf64 asmfunc1.asm -o asmfunc1.o
!nasm -f elf64 asmfunc2.asm -o asmfunc2.o
!nasm -f elf64 asmfunc3.asm -o asmfunc3.o

!gcc -O3 -fno-tree-vectorize main.c asmfunc1.o asmfunc2.o asmfunc3.o -o benchmark

In [6]:
!./benchmark


N = 1048576 elements

C Kernel
First 5 elements:
[0] C = 999.00	idx = 1
[1] C = 998.00	idx = 1
[2] C = 997.00	idx = 1
[3] C = 996.00	idx = 1
[4] C = 995.00	idx = 1

Last 5 elements:
[1048571] C = 571.00	idx = 0
[1048572] C = 572.00	idx = 0
[1048573] C = 573.00	idx = 0
[1048574] C = 574.00	idx = 0
[1048575] C = 575.00	idx = 0

Scalar : PASS

Scalar
First 5 elements:
[0] C = 999.00	idx = 1
[1] C = 998.00	idx = 1
[2] C = 997.00	idx = 1
[3] C = 996.00	idx = 1
[4] C = 995.00	idx = 1

Last 5 elements:
[1048571] C = 571.00	idx = 0
[1048572] C = 572.00	idx = 0
[1048573] C = 573.00	idx = 0
[1048574] C = 574.00	idx = 0
[1048575] C = 575.00	idx = 0

XMM    : PASS

XMM
First 5 elements:
[0] C = 999.00	idx = 1
[1] C = 998.00	idx = 1
[2] C = 997.00	idx = 1
[3] C = 996.00	idx = 1
[4] C = 995.00	idx = 1

Last 5 elements:
[1048571] C = 571.00	idx = 0
[1048572] C = 572.00	idx = 0
[1048573] C = 573.00	idx = 0
[1048574] C = 574.00	idx = 0
[1048575] C = 575.00	idx = 0

YMM    : PASS

YMM
First 5 elements: